In [0]:
%pip install websockets azure-eventhub

In [0]:
import asyncio
import json
import logging
from websockets import connect
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData

CONNECTION_STR = dbutils.secrets.get(
    scope="trading_secrets",
    key="eventhub_connection_string"
)

EVENTHUB_NAME  = "klines-raw"
BATCH_SIZE     = 50
RETRY_DELAY_S  = 5

SYMBOLS = [
    'btcusdt', 'ethusdt', 'solusdt', 'bnbusdt',  'xrpusdt',
    'adausdt', 'avaxusdt','dotusdt', 'linkusdt', 'dogeusdt',
]

STREAMS        = "/".join([f"{s}@kline_1m" for s in SYMBOLS])
BINANCE_WS_URL = f"wss://stream.binance.com:9443/stream?streams={STREAMS}"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("Producer")


async def produce():
    client = EventHubProducerClient.from_connection_string(
        conn_str=CONNECTION_STR,
        eventhub_name=EVENTHUB_NAME
    )

    async with client:
        while True:
            batch = None
            try:
                logger.info("Connecting to Binance WebSocket...")
                async with connect(
                    BINANCE_WS_URL,
                    ping_interval=30,
                    ping_timeout=10,
                ) as websocket:
                    logger.info("Connection established.")
                    batch = await client.create_batch()

                    async for message in websocket:
                        msg  = json.loads(message)
                        data = msg.get("data")

                        if not data or "k" not in data:
                            logger.warning(f"Unexpected format: {message[:100]}")
                            continue

                        k         = data["k"]
                        is_closed = k.get("x", False)

                        event_data = EventData(json.dumps(data))
                        try:
                            batch.add(event_data)
                        except ValueError:
                            await client.send_batch(batch)
                            logger.info(f"Batch size limit — flushed and reset")
                            batch = await client.create_batch()
                            batch.add(event_data)

                        if len(batch) >= BATCH_SIZE:
                            await client.send_batch(batch)
                            logger.info(
                                f"Sent {BATCH_SIZE} messages | "
                                f"symbol={k.get('s')} | "
                                f"closed={is_closed}"
                            )
                            batch = await client.create_batch()

            except Exception as e:
                logger.error(f"Error: {e}. Reconnecting in {RETRY_DELAY_S}s...")
                if batch and len(batch) > 0:
                    try:
                        await client.send_batch(batch)
                        logger.info(f"Flushed remaining messages before reconnect")
                    except Exception as flush_err:
                        logger.error(f"Flush failed: {flush_err}")

                await asyncio.sleep(RETRY_DELAY_S)

await produce()